<center> <h1>Google Maps "Grounding Lite" MCP - Overview & Usage</h1>

### 0. Prep

In [ ]:
%pip install --upgrade langchain-mcp-adapters

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### 1. MCP Client Setup

In [2]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "gmaps-mcp": {
            "url": f"https://mapstools.googleapis.com/mcp",
            "transport": "streamable_http",
            "headers": {
                "x-goog-api-key": os.getenv("GOOGLE_MAPS_API_KEY")
            }
        }
    }
)

Check the tools exposed by the server :

In [3]:
tools = await client.get_tools()
tools

[StructuredTool(name='search_places', description='\nCall this tool when the user\'s request is to find places, businesses, addresses, locations, points of interest, or any other Google Maps related search.\n\n**Input Requirements (CRITICAL):**\n\n1.  **`text_query` (string - MANDATORY):** The primary search query. This must clearly define what the user is looking for.\n\n    *   **Examples:** `\'restaurants in New York\'`, `\'coffee shops near Golden Gate Park\'`, `\'SF MoMA\'`, `\'1600 Amphitheatre Pkwy, Mountain View, CA, USA\'`, `\'pets friendly parks in Manhattan, New York\'`, `\'date night restaurants in Chicago\'`, `\'accessible public libraries in Los Angeles\'`.\n\n    *   **For specific place details:** Include the requested attribute (e.g., `\'Google Store Mountain View opening hours\'`, `\'SF MoMa phone number\'`, `\'Shoreline Park Mountain View address\'`).\n\n2.  **`location_bias` (object - OPTIONAL):** Use this to prioritize results near a specific geographic area.\n    

In [4]:
search_places_tool = tools[0]
lookup_weather_tool = tools[1]
compute_routes_tool = tools[2]

### 2. Search Places

#### 2.1 Input Schema

In [5]:
import json
print(json.dumps(search_places_tool.args_schema, indent=4))

{
    "$defs": {
        "Circle": {
            "description": "A circle defined by center point and radius.",
            "properties": {
                "center": {
                    "$ref": "#/$defs/LatLng",
                    "description": "Required. The center point of the circle."
                },
                "radiusMeters": {
                    "description": "The radius of the circle in meters. The radius must be within 50,000 meters.",
                    "format": "double",
                    "type": "number"
                }
            },
            "required": [
                "center"
            ],
            "type": "object"
        },
        "LatLng": {
            "description": "An object that represents a latitude/longitude pair. This is expressed as a pair of doubles to represent degrees latitude and degrees longitude. Unless specified otherwise, this object must conform to the WGS84 standard <https://en.wikipedia.org/wiki/World_Geodetic_System#19

#### 2.2 Usage

##### A. Specific query in natural language

In [6]:
places = await search_places_tool.ainvoke({"textQuery": "coffee shops in paris"})

In [7]:
print(places)

{
  "places": [
    {
      "place": "places/ChIJ3eCHM_1x5kcRRfrf5-yoThc",
      "id": "ChIJ3eCHM_1x5kcRRfrf5-yoThc",
      "location": {
        "latitude": 48.8544525,
        "longitude": 2.3557666999999998
      },
      "googleMapsLinks": {
        "directionsUrl": "https://www.google.com/maps/dir//''/data=!4m7!4m6!1m1!4e2!1m2!1m1!1s0x47e671fd3387e0dd:0x174ea8ece7dffa45!3e0",
        "placeUrl": "https://maps.google.com/?cid=1679465446511737413",
        "writeAReviewUrl": "https://www.google.com/maps/place//data=!4m3!3m2!1s0x47e671fd3387e0dd:0x174ea8ece7dffa45!12e1",
        "reviewsUrl": "https://www.google.com/maps/place//data=!4m4!3m3!1s0x47e671fd3387e0dd:0x174ea8ece7dffa45!9m1!1b1",
        "photosUrl": "https://www.google.com/maps/place//data=!4m3!3m2!1s0x47e671fd3387e0dd:0x174ea8ece7dffa45!10e5"
      }
    },
    {
      "place": "places/ChIJEapuZOVv5kcRua-GIPTaSC0",
      "id": "ChIJEapuZOVv5kcRua-GIPTaSC0",
      "location": {
        "latitude": 48.8567259,
        "lon

In [9]:
print(json.loads(places)["summary"])

Here are some coffee shops in Paris:

**The Caféothèque of Paris** is located at 52 Rue de l'Hôtel de Ville, 75004 Paris, France. It is open from Monday to Sunday, with hours varying slightly on Sundays. They serve breakfast, lunch, beer, and wine, and offer outdoor seating. [0]

**Café du Marais** is located at 3 Rue du Bourg Tibourg, 75004 Paris, France. They are open late every day of the week and serve breakfast, lunch, dinner, beer, wine, and brunch. They also offer outdoor seating and are good for children. [1]

**Clove** is located at 14 Rue Chappe, 75018 Paris, France. They are open from 9:00 AM to 4:00 PM on Sundays, Thursdays, Fridays, Saturdays, and Mondays, and are closed on Tuesdays and Wednesdays. They serve dessert and coffee. [2]

**Brouillon Coffee** is located at 42 Bd de Magenta, 75010 Paris, France. They are open from 8:00 AM to 4:30 PM from Monday to Friday and are closed on Saturdays and Sundays. They serve breakfast and dessert, and offer outdoor seating. They ar

##### B. General query + specific search area (location bias)

In [10]:
places_bias = await search_places_tool.ainvoke({
    "textQuery": "coffee shops",
    "locationBias": {
        "circle": {
            "center": {"latitude": 52.37308, "longitude": 4.892453},
            "radiusMeters": 5000
        }
    }
})

In [11]:
print(places_bias)

{
  "places": [
    {
      "place": "places/ChIJWzS1LckJxkcR1ht8nU7G_eY",
      "id": "ChIJWzS1LckJxkcR1ht8nU7G_eY",
      "location": {
        "latitude": 52.3806892,
        "longitude": 4.8908553
      },
      "googleMapsLinks": {
        "directionsUrl": "https://www.google.com/maps/dir//''/data=!4m7!4m6!1m1!4e2!1m2!1m1!1s0x47c609c92db5345b:0xe6fdc64e9d7c1bd6!3e0",
        "placeUrl": "https://maps.google.com/?cid=16644677838783126486",
        "writeAReviewUrl": "https://www.google.com/maps/place//data=!4m3!3m2!1s0x47c609c92db5345b:0xe6fdc64e9d7c1bd6!12e1",
        "reviewsUrl": "https://www.google.com/maps/place//data=!4m4!3m3!1s0x47c609c92db5345b:0xe6fdc64e9d7c1bd6!9m1!1b1",
        "photosUrl": "https://www.google.com/maps/place//data=!4m3!3m2!1s0x47c609c92db5345b:0xe6fdc64e9d7c1bd6!10e5"
      }
    },
    {
      "place": "places/ChIJTz9j_cgJxkcRHmTGGEXPMPw",
      "id": "ChIJTz9j_cgJxkcRHmTGGEXPMPw",
      "location": {
        "latitude": 52.3800848,
        "longitude":

In [12]:
print(json.loads(places_bias)["summary"])

Here are some coffee shops in Amsterdam:

**Barney's Coffeeshop Amsterdam THC Cannabis Dispensary** is located at Haarlemmerstraat 102, 1013 EW Amsterdam, Netherlands. It has a rating of 4.6 stars based on 4938 reviews and is operational [0]. They accept debit cards and NFC payments [0].

**Coffeeshop Amsterdam** is located at Haarlemmerstraat 44, 1013 ES Amsterdam, Netherlands. This coffee shop has a rating of 4.5 stars from 2344 reviews and is operational [1]. They accept credit cards, debit cards, and NFC payments [1].

**Cafe the Barrel** is located at Jonge Roelensteeg 4 H, 1012 PL Amsterdam, Netherlands. It has a rating of 4.8 stars from 374 reviews and is operational [2]. This establishment serves beer, wine, and coffee, and allows dogs [2]. They accept credit cards, debit cards, and NFC payments [2].

**De Kroon Coffeeshop Amsterdam** is located at Oudebrugsteeg 26, 1012 JP Amsterdam, Netherlands. It has a rating of 4.6 stars from 1870 reviews and is operational [3]. They accep

### 3. Lookup Weather

#### 3.1 Input Schema

In [10]:
print(json.dumps(lookup_weather_tool.args_schema, indent=4))

{
    "$defs": {
        "Date": {
            "description": "Represents a whole or partial calendar date, such as a birthday. The time of day and time zone are either specified elsewhere or are insignificant. The date is relative to the Gregorian Calendar. This can represent one of the following: * A full date, with non-zero year, month, and day values. * A month and day, with a zero year (for example, an anniversary). * A year on its own, with a zero month and a zero day. * A year and month, with a zero day (for example, a credit card expiration date). Related types: * google.type.TimeOfDay * google.type.DateTime * google.protobuf.Timestamp",
            "properties": {
                "day": {
                    "description": "Day of a month. Must be from 1 to 31 and valid for the year and month, or 0 to specify a year by itself or a year and month where the day isn't significant.",
                    "format": "int32",
                    "type": "integer"
                },
  

#### 3.2 Usage

##### A. Precise location & time

In [16]:
weather = await lookup_weather_tool.ainvoke({
    "location": {
        "latLng": {"latitude": 48.8566, "longitude": 2.3522}
    },
    "date": {
        "year": 2026, 
        "month": 1, 
        "day": 22 # Future 7 days only
    },
    "hour": 16, # Future 48 hours only
})

In [17]:
print(weather)

{
  "temperature": {
    "degrees": 10.3,
    "unit": "CELSIUS"
  },
  "feelsLikeTemperature": {
    "degrees": 8,
    "unit": "CELSIUS"
  },
  "heatIndex": {
    "degrees": 10.3,
    "unit": "CELSIUS"
  },
  "airPressure": {
    "meanSeaLevelMillibars": 992.41
  },
  "weatherCondition": {
    "iconBaseUri": "https://maps.gstatic.com/weather/v1/partly_cloudy",
    "description": {
      "text": "Partly sunny",
      "languageCode": "en"
    },
    "type": "PARTLY_CLOUDY"
  },
  "precipitation": {
    "probability": {
      "percent": 15,
      "type": "RAIN"
    },
    "snowQpf": {
      "quantity": 0,
      "unit": "MILLIMETERS"
    },
    "qpf": {
      "quantity": 0,
      "unit": "MILLIMETERS"
    }
  },
  "wind": {
    "direction": {
      "degrees": 195,
      "cardinal": "SOUTH_SOUTHWEST"
    },
    "speed": {
      "value": 16,
      "unit": "KILOMETERS_PER_HOUR"
    },
    "gust": {
      "value": 35,
      "unit": "KILOMETERS_PER_HOUR"
    }
  },
  "relativeHumidity": 81,
  "

##### B. General location + wider time period + Imperial units system

In [20]:
weather_imperial = await lookup_weather_tool.ainvoke({
    "location": {
        "address": "Paris, France"
    },
    "date": {
        "year": 2026,
        "month": 1,
        "day": 22 # Future 7 days only
    },
    "unitsSystem": "IMPERIAL",
})

In [21]:
print(weather_imperial)

{
  "maxTemperature": {
    "degrees": 50.9,
    "unit": "FAHRENHEIT"
  },
  "minTemperature": {
    "degrees": 43.1,
    "unit": "FAHRENHEIT"
  },
  "feelsLikeMaxTemperature": {
    "degrees": 46.4,
    "unit": "FAHRENHEIT"
  },
  "feelsLikeMinTemperature": {
    "degrees": 39.2,
    "unit": "FAHRENHEIT"
  },
  "maxHeatIndex": {
    "degrees": 50.9,
    "unit": "FAHRENHEIT"
  },
  "sunEvents": {
    "sunriseTime": "2026-01-22T07:32:45.203172472Z",
    "sunsetTime": "2026-01-22T16:32:04.088537135Z"
  },
  "moonEvents": {
    "moonPhase": "WAXING_CRESCENT",
    "moonriseTimes": [
      "2026-01-22T09:11:42.854325762Z"
    ],
    "moonsetTimes": [
      "2026-01-22T20:53:53.500025635Z"
    ]
  },
  "weatherCondition": {
    "iconBaseUri": "https://maps.gstatic.com/weather/v1/partly_cloudy",
    "description": {
      "text": "Partly sunny",
      "languageCode": "en"
    },
    "type": "PARTLY_CLOUDY"
  },
  "precipitation": {
    "probability": {
      "percent": 20,
      "type": "RAIN

### 4. Compute Routes

#### 4.1 Input Schema

In [22]:
print(json.dumps(compute_routes_tool.args_schema, indent=4))

{
    "$defs": {
        "LatLng": {
            "description": "An object that represents a latitude/longitude pair. This is expressed as a pair of doubles to represent degrees latitude and degrees longitude. Unless specified otherwise, this object must conform to the WGS84 standard <https://en.wikipedia.org/wiki/World_Geodetic_System#1984_version>. Values must be within normalized ranges.",
            "properties": {
                "latitude": {
                    "description": "The latitude in degrees. It must be in the range [-90.0, +90.0].",
                    "format": "double",
                    "type": "number"
                },
                "longitude": {
                    "description": "The longitude in degrees. It must be in the range [-180.0, +180.0].",
                    "format": "double",
                    "type": "number"
                }
            },
            "type": "object"
        },
        "Waypoint": {
            "description": "Encapsulat

#### 4.2 Usage

##### A. Walk + Addresses

In [23]:
route = await compute_routes_tool.ainvoke({
    "origin": {
        "address": "Madison Square Garden, New York, NY"
    },
    "destination": {
        "address": "Central Park, New York, NY"
    },
    "travelMode": "WALK"
})

In [24]:
print(route)

{
  "routes": [
    {
      "distanceMeters": 2185,
      "duration": "1899s"
    }
  ]
}



##### B. Drive + Address + LatLong

In [25]:
route_drive = await compute_routes_tool.ainvoke({
    "origin": {
        "address": "Rotterdam, Netherlands"
    },
    "destination": {
        "latLng": {"latitude": 52.37308, "longitude": 4.892453}
    },
    "travelMode": "DRIVE",
})

In [26]:
print(route_drive)

{
  "routes": [
    {
      "distanceMeters": 79241,
      "duration": "3961s"
    }
  ]
}

